In [3]:
!pip install -q transformers datasets accelerate

In [10]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Configuration
SUBSET_SIZE = 50000 
MAX_LENGTH = 128
MODEL_NAME = "bert-base-uncased"

# 2. Data Preparation
print("Loading and subsetting dataset...")
dataset = load_dataset("fancyzhx/amazon_polarity")
train_val_subset = dataset['train'].shuffle(seed=42).select(range(SUBSET_SIZE))
test_subset = dataset['test'].shuffle(seed=42).select(range(int(SUBSET_SIZE * 0.2))) 
train_val_split = train_val_subset.train_test_split(test_size=0.2, seed=42)

print("Tokenizing...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    texts = [f"{t} {c}" for t, c in zip(examples['title'], examples['content'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_val_split['train'].map(tokenize_function, batched=True, remove_columns=['title', 'content'])
tokenized_val = train_val_split['test'].map(tokenize_function, batched=True, remove_columns=['title', 'content'])

# 3. Model Initialization (Training Mode)
print("Initializing model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
    # output_attentions=True is REMOVED to save massive VRAM during training.
    # We will turn it back on during the evaluation/explainability script!
)

# 4. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

# 5. Training Setup
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,  # Back to 16 for faster training
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Mixed precision enabled
    report_to="none" 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Starting training on GPU...")
trainer.train()

print("\nEvaluating on validation set...")
eval_results = trainer.evaluate()
print(f"\nFinal Validation Metrics:\n{eval_results}")

print("\nSaving best model...")
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")
print("Done! You can now download the './best_model' folder.")

Loading and subsetting dataset...
Tokenizing...
Initializing model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training on GPU...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on validation set...



Final Validation Metrics:
{'eval_loss': 0.5126388669013977, 'eval_accuracy': 0.9476, 'eval_precision': 0.9497017892644135, 'eval_recall': 0.9463153724247226, 'eval_f1': 0.9480055566580671, 'eval_runtime': 51.6005, 'eval_samples_per_second': 193.797, 'eval_steps_per_second': 6.066, 'epoch': 3.0}

Saving best model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done! You can now download the './best_model' folder.


In [11]:
import shutil
from IPython.display import FileLink

print("Zipping the best_model directory...")
# This compresses the folder into a single file named 'best_model.zip'
shutil.make_archive('best_model', 'zip', './best_model')

print("Zip complete! Click the link below to download:")
# This generates a clickable download link directly in the notebook output
FileLink(r'best_model.zip')

Zipping the best_model directory...
Zip complete! Click the link below to download:


/kaggle/working/best_model.zip